In [1]:
import pandas as pd
import glob
import datetime
from zoneinfo import ZoneInfo
import exchange_calendars as ec
import torch
import os
import numpy as np

In [2]:

def is_full_nyse_day(date):
    nyse = ec.get_calendar("XNYS")
    date = pd.Timestamp(date).normalize()

    if not nyse.is_session(date):
        return False

    open_time   = nyse.session_open(date)
    close_time  = nyse.session_close(date)
    full_length = pd.Timedelta(hours=6.5)
    return (close_time - open_time) >= full_length


def get_full_sequence_data_by_day():

    folder_path      = r"../../../MarketData/historical_data"
    files            = glob.glob(os.path.join(folder_path, "spy_1min_*.csv"))

    xs = []

    for file in files:

        date_str = file.split("_")[-1].split(".")[0]
        print(date_str)
        date_str = date_str[:4] + "-" + date_str[4:6] + "-" + date_str[6:]

        if not is_full_nyse_day(date_str):
            print("skipping", date_str)
            continue

        df = pd.read_csv(file)

        open_time = datetime.datetime.strptime(date_str + " 09:31:00", "%Y-%m-%d %H:%M:%S").replace(tzinfo= ZoneInfo("America/New_York"))
        close_time = datetime.datetime.strptime(date_str + " 16:00:00", "%Y-%m-%d %H:%M:%S").replace(tzinfo= ZoneInfo("America/New_York"))

        df["ts_recv"] = pd.to_datetime(df["ts_recv"])
        df['ts_recv_est'] = df['ts_recv'].dt.tz_convert('America/New_York')
        df = df[(df["ts_recv_est"] >= open_time) & (df["ts_recv_est"] <= close_time)]
        df["seconds_since_open"] = (df["ts_recv_est"] - open_time).dt.total_seconds()
        data = df[["ret_60s"]] #, "rv_60s", "seconds_since_open"]].values

        data = np.array(data)
        xs.append(data)

    xs = np.array(xs)
    return  xs


In [3]:
returns = get_full_sequence_data_by_day()
returns = torch.from_numpy(returns)   # shape (1246, 390)
print(returns.shape)

20201005
20201006
20201007
20201008
20201009
20201012
20201013
20201014
20201015
20201016
20201019
20201020
20201021
20201022
20201023
20201026
20201027
20201028
20201029
20201030
20201102
20201103
20201104
20201105
20201106
20201109
20201110
20201111
20201112
20201113
20201116
20201117
20201118
20201119
20201120
20201123
20201124
20201125
20201127
skipping 2020-11-27
20201130
20201201
20201202
20201203
20201204
20201207
20201208
20201209
20201210
20201211
20201214
20201215
20201216
20201217
20201218
20201221
20201222
20201223
20201224
skipping 2020-12-24
20201228
20201229
20201230
20201231
20210104
20210105
20210106
20210107
20210108
20210111
20210112
20210113
20210114
20210115
20210119
20210120
20210121
20210122
20210125
20210126
20210127
20210128
20210129
20210201
20210202
20210203
20210204
20210205
20210208
20210209
20210210
20210211
20210212
20210216
20210217
20210218
20210219
20210222
20210223
20210224
20210225
20210226
20210301
20210302
20210303
20210304
20210305
20210308
202103

In [4]:
import torch

num_days, num_minutes, _ = returns.shape
context = 60
horizon = 60

Xs = []
ys = []

for d in range(num_days):
    day = returns[d]                      # shape (390,)
    for t in range(context, num_minutes-horizon):
        Xs.append(day[t-context:t])       # (60,)
        ys.append(day[t: t+ horizon])     # (60,)

Xs = torch.stack(Xs)     # (1246*(390-60), 60, 1)
ys = torch.stack(ys)     # (1246*(390-60), 60, 1)
print(Xs.shape, ys.shape)

torch.Size([336420, 60, 1]) torch.Size([336420, 60, 1])


In [5]:
N = Xs.shape[0]

train_end = int(0.6 * N)
val_end   = int(0.8 * N)

Xs_train_raw = Xs[:train_end]
ys_train_raw = ys[:train_end]

Xs_val_raw = Xs[train_end:val_end]
ys_val_raw = ys[train_end:val_end]

Xs_test_raw = Xs[val_end:]
ys_test_raw = ys[val_end:]

mean_train = Xs_train_raw.mean()
std_train  = Xs_train_raw.std()

Xs_train = (Xs_train_raw - mean_train) / std_train
Xs_val   = (Xs_val_raw   - mean_train) / std_train
Xs_test  = (Xs_test_raw  - mean_train) / std_train

ys_train = (ys_train_raw - mean_train) / std_train
ys_val   = (ys_val_raw   - mean_train) / std_train
ys_test  = (ys_test_raw  - mean_train) / std_train

print(Xs_train.shape, ys_train.shape)
print(ys_train.shape, ys_val.shape)

print(Xs_val.shape, Xs_val.shape)
print(ys_val.shape, ys_val.shape)

print(Xs_test.shape, Xs_test.shape)
print(ys_test.shape, ys_test.shape)

torch.Size([201852, 60, 1]) torch.Size([201852, 60, 1])
torch.Size([201852, 60, 1]) torch.Size([67284, 60, 1])
torch.Size([67284, 60, 1]) torch.Size([67284, 60, 1])
torch.Size([67284, 60, 1]) torch.Size([67284, 60, 1])
torch.Size([67284, 60, 1]) torch.Size([67284, 60, 1])
torch.Size([67284, 60, 1]) torch.Size([67284, 60, 1])


In [6]:
import torch.nn as nn
import torch.nn.functional as F

class LSTMNormal_M2M(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True
        )
        self.mu_head = nn.Linear(hidden, 1)
        self.sigma_head = nn.Linear(hidden, 1)

    def forward(self, x):

        out, _ = self.lstm(x)
        h =  out[:, :, :]  # use all 60 predictions
        batch_size, seq_len, _ = h.shape
        mu = torch.zeros((batch_size, seq_len, 1), device=x.device)
        sigma  = F.softplus(self.sigma_head(h)) + 1e-6

        return mu, sigma

def nll_normal_m2m(y, mu, sigma):
    return (
        0.5 * torch.log(2 * torch.pi * sigma**2) +
        (y - mu)**2 / (2 * sigma**2)
    ).mean()


In [7]:
from torch.utils.data import Dataset, DataLoader

class ReturnDataset(Dataset):
    def __init__(self, X, y):
        self.X = X   # (N, 60, 1)
        self.y = y   # (N, 60, 1)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = ReturnDataset(Xs_train, ys_train)
val_ds   = ReturnDataset(Xs_val,   ys_val)
test_ds  = ReturnDataset(Xs_test,  ys_test)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)


In [ ]:
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loss_history      = []
sigma_history     = []

val_loss_history  = []
val_sigma_history = []

model = LSTMNormal_M2M().to(device).double()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(300):

    model.train()
    epoch_losses = []
    epoch_sigmas = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device).double(), yb.to(device).double()

        mu, sigma = model(xb)
        loss = nll_normal_m2m(yb, mu, sigma)

        epoch_losses.append(loss.item())

        sigma_step_means = sigma.mean(dim=0).squeeze(-1)  # (60,)
        epoch_sigmas.append(sigma_step_means.detach().cpu().numpy())

        opt.zero_grad()
        loss.backward()
        opt.step()

    loss_history.append(np.mean(epoch_losses))
    sigma_history.append(np.mean(epoch_sigmas, axis=0))

    model.eval()
    val_losses = []
    val_sigmas = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device).double()
            yb = yb.to(device).double()

            mu, sigma = model(xb)
            val_loss = nll_normal_m2m(yb, mu, sigma)

            val_losses.append(val_loss.item())
            sigma_step_means = sigma.mean(dim=0).squeeze(-1)  # (60,)
            val_sigmas.append(sigma_step_means.detach().cpu().numpy())

    val_loss_history.append(np.mean(val_losses))
    val_sigma_history.append( np.mean(val_sigmas, axis=0))

    print(f"Epoch {epoch}: train loss={loss_history[-1]:.4f}, val loss={val_loss_history[-1]:.4f}")

Epoch 0: train loss=1.2774, val loss=0.9963
Epoch 1: train loss=1.2395, val loss=0.9966
Epoch 2: train loss=1.2336, val loss=1.0007
Epoch 3: train loss=1.2309, val loss=1.0066
Epoch 4: train loss=1.2265, val loss=1.0042
Epoch 5: train loss=1.2220, val loss=1.0086
Epoch 6: train loss=1.2180, val loss=1.0012
Epoch 7: train loss=1.2162, val loss=0.9981
Epoch 8: train loss=1.2117, val loss=0.9967
Epoch 9: train loss=1.2120, val loss=0.9908
Epoch 10: train loss=1.2070, val loss=0.9948
Epoch 11: train loss=1.2034, val loss=1.0056
Epoch 12: train loss=1.2006, val loss=1.0033
Epoch 13: train loss=1.1994, val loss=1.0012
Epoch 14: train loss=1.1956, val loss=1.0040
Epoch 15: train loss=1.1929, val loss=1.0056
Epoch 16: train loss=1.1894, val loss=1.0037
Epoch 17: train loss=1.1868, val loss=1.0085
Epoch 18: train loss=1.1839, val loss=1.0107
Epoch 19: train loss=1.1807, val loss=1.0074
Epoch 20: train loss=1.1783, val loss=1.0047
Epoch 21: train loss=1.1754, val loss=1.0136
Epoch 22: train loss

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12,5))
plt.plot(loss_history, label="Train Loss", linewidth=2)
plt.plot(val_loss_history, label="Validation Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss (NLL)")
plt.title("Train vs Validation Loss - LSTM + Normal (60→60)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
sigma_hist     = np.array(sigma_history)      # (epochs, 60)
val_sigma_hist = np.array(val_sigma_history)  # (epochs, 60)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sigma_hist = np.array(sigma_history)   # shape: (epochs, 60)

plt.figure(figsize=(14, 7))

for h in range(60):
    plt.plot(
        sigma_hist[:, h],
        linewidth=1,
        alpha=0.6
    )

plt.xlabel("Epoch")
plt.ylabel("Sigma")
plt.title("Train Sigma  (average cross samples) — 60 Horizons")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(14, 7))

for h in range(60):
    plt.plot(
        val_sigma_hist[:, h],
        linewidth=1,
        alpha=0.6
    )

plt.xlabel("Epoch")
plt.ylabel("Sigma")
plt.title("Validation Sigma (average cross samples) — All 60 Future Horizons (LSTM Normal 60→60)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Model:
: $$x_{t+1}, x_{t+2}, \cdots, x_{t+60}|{x_{t-59}, x_{t-58},\cdots, x_t} \sim  \mathcal{N}(0, Diag((\sigma_{t}^{(1)})^2, (\sigma_{t}^{(2)})^2, \cdots, (\sigma_{t}^{(60)})^2)$$